### Demo / Tests anhand Gemeinde Rifferswil

In [ ]:
import folium
import json
from urllib.request import urlopen

# URL zur Gemeinde Rifferswil (BFS-Nr 12)
url = "https://api3.geo.admin.ch/rest/services/api/MapServer/ch.swisstopo.swissboundaries3d-gemeinde-flaeche.fill/12?geometryFormat=geojson&sr=4326"

# GeoJSON herunterladen
with urlopen(url) as response:
    geojson_data = json.load(response)

# Mittelpunkt aus bbox berechnen
bbox = geojson_data["feature"]["bbox"]
center_lat = (bbox[1] + bbox[3]) / 2
center_lon = (bbox[0] + bbox[2]) / 2

# Karte erstellen
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles="CartoDB positron"
)

# GeoJSON hinzufügen
folium.GeoJson(
    
    geojson_data["feature"],
    name="Gemeinde Rifferswil",
    style_function=lambda x: {
        "fillColor": "#3186cc",
        "color": "black",
        "weight": 2,
        "fillOpacity": 0.5,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["gemname", "gde_nr", "jahr"],
        aliases=["Gemeinde", "BFS-Nr", "Jahr"]
    )

).add_to(m)

# Layer Control
folium.LayerControl().add_to(m)

# Speichern als HTML
file_path = "rifferswil_map.html"
m.save(file_path)

file_path

'rifferswil_map.html'

In [3]:
import os
if os.path.exists(file_path):
    os.remove(file_path)
    print("Datei wurde gelöscht.")
else:
    print("Datei existiert nicht.")

Datei wurde gelöscht.


### 1. Steckbriefe für Gemeinden

TBD: 

- Link zu Bezirk

In [14]:
import json
import os
from urllib.request import urlopen
import folium
import requests
import xml.etree.ElementTree as ET

# -----------------------------
# 1. Gemeindezuweisungen laden
# -----------------------------
zuweisungen_url = "https://gebietsstammdaten.statistik.zh.ch/api/gemeindezuweisungen"

with urlopen(zuweisungen_url) as response:
    zuweisungen_data = json.load(response)
    
    gemeinden = zuweisungen_data["gemeindezuweisungen"]
    
    output_dir = "../../docs/gemeinden/"
    os.makedirs(output_dir, exist_ok=True)
    
    # Logo-Dateiname
    logo_filename = "../logo.svg"
    
# -----------------------------
# 2. Für jede Gemeinde Steckbrief erzeugen
# -----------------------------
for g in gemeinden:
    
    #Stammdaten laden
    gemeinde_code = g["gemeinde_code"]
    gemeinde_name = g["gemeinde_name"]
    
    print(f"Erstelle Steckbrief für {gemeinde_name}...")

    safe_name = gemeinde_code


    #Geodaten laden
    filter_encoded = f"%3CFilter%3E%3CPropertyIsEqualTo%3E%3CPropertyName%3Ebfs%3C/PropertyName%3E%3CLiteral%3E{gemeinde_code}%3C/Literal%3E%3C/PropertyIsEqualTo%3E%3C/Filter%3E"

    base_wfs = "https://maps.zh.ch/wfs/OGDZHWFS"

    geojson_link = f"{base_wfs}?SERVICE=WFS&VERSION=2.0.0&REQUEST=GetFeature&TYPENAMES=ms:ogd-0095_arv_basis_up_gemeinden_f&outputFormat=geojson&FILTER={filter_encoded}"

    gml_link = f"{base_wfs}?SERVICE=WFS&VERSION=2.0.0&REQUEST=GetFeature&TYPENAMES=ms:ogd-0095_arv_basis_up_gemeinden_f&outputFormat=gml3&FILTER={filter_encoded}"

    #Zentrierung anhand Koordinaten im GML für Link aufs Geoportal ZH erzeugen
    response = requests.get(gml_link)
    tree = ET.fromstring(response.content)

    ns = {
        "wfs": "http://www.opengis.net/wfs/2.0",
        "gml": "http://www.opengis.net/gml",
        "ms": "http://mapserver.gis.umn.edu/mapserver"
    }

    # gml parsen
    envelope = tree.find(".//wfs:boundedBy/gml:Envelope", ns)
    lower_corner = envelope.find("gml:lowerCorner", ns).text
    upper_corner = envelope.find("gml:upperCorner", ns).text

    lower_x, lower_y = map(float, lower_corner.split())
    upper_x, upper_y = map(float, upper_corner.split())

    x_center = ((lower_x + upper_x) / 2) - 1000 #Minus tausend 1000, aufgrund der Legende links. So wird es im Kartenbereich mittig.
    y_center = (lower_y + upper_y) / 2

    scale = 20000

    gis_link = f"https://geo.zh.ch/maps?x={x_center}&y={y_center}&scale={scale}&basemap=arelkbackgroundzh"
    
    # -----------------------------
    # Steckbrief HTML
    # -----------------------------
    html_content = f"""
<!DOCTYPE html>
<html lang="de">
<head>
<meta charset="UTF-8">
<title>{gemeinde_name}</title>
<style>
body {{
    font-family: Arial, sans-serif;
    margin: 0;
    background-color: #f4f6f9;
}}

.header {{
    background-color: #0076bd;
    color: white;
    padding: 20px 40px;
    display: flex;
    align-items: center;
}}

.header img {{
    height: 60px;
    margin-right: 20px;
    filter: brightness(0) invert(1);
}}

.header h1 {{
    margin: 0;
}}

.content {{
    padding: 40px;
}}

.card {{
    background: white;
    padding: 25px;
    border-radius: 12px;
    box-shadow: 0 4px 12px rgba(0,0,0,0.08);
    margin-bottom: 30px;
}}

.card h2 {{
    color: #0076bd;
    margin-top: 0;
}}

iframe {{
    width: 100%;
    height: 500px;
    border: none;
    border-radius: 12px;
}}

.download-btn {{
    background-color: #0076bd;
    color: white;
    padding: 12px 18px;
    border-radius: 6px;
    text-decoration: none;
    margin-right: 15px;
    display: inline-block;
}}

.download-btn:hover {{
    background-color: #0076bd;
}}
</style>
</head>

<body>

<div class="header">
    <img src="../logo.svg" alt="Zürich Leu">
    <h1>{gemeinde_name}</h1>
</div>

<div class="content">

<div class="card">
    <p><strong>Gemeinde-Code (Bfsnr): </strong>{gemeinde_code}</p>
    <p><strong>Bezirk: </strong>{g["bezirk_name"]} ({g["bezirk_code"]})</p>
    <p><strong>Raumplanungsregion: </strong>{g["raumplanungsregion_name"]} ({g["raumplanungsregion_code"]})</p>
</div>

<div class="card">
    <h2>Lage</h2>
    <iframe src="https://map.geo.admin.ch/embed.html?ch.swisstopo.swissboundaries3d-gemeinde-flaeche.fill={gemeinde_code}"></iframe>
</div>

<div class="card">
    <h2>Geodaten</h2>
    <a class="download-btn" href="{gis_link}" target="_blank">Geoportal Kanton Zürich</a>
   <a class="download-btn" href="{geojson_link}" target="_blank">GeoJSON (WFS)</a>
<a class="download-btn" href="{gml_link}" target="_blank">GML</a>
</div>

<div class="card">
    <h2>Datenquellen / Links</h2>
    <p>    
        Gemeinden und Bezirke: 
        <a href="https://www.bfs.admin.ch/bfs/de/home/grundlagen/agvch.html" target="_blank">BFS – Amtliches Gemeindenverzeichnis</a><br>
        Raumplanungsregionen: 
        <a href="https://www.bfs.admin.ch/bfs/de/home/statistiken/querschnittsthemen/raeumliche-analysen/raeumliche-gliederungen/regionalpolitische-gliederungen.html" target="_blank">Raumplanungsregionen</a><br>
        Gebietsstammdaten ZH: 
        <a href="https://www.zh.ch/de/politik-staat/statistik-daten/datenkatalog.html#/datasets/3082@statistisches-amt-kanton-zuerich" target="_blank">Datenkatalog ZH</a><br>
Geoportal Kanton Zürich: 
<a href="https://geo.zh.ch/maps?x=2682088&y=1253620&scale=282230&basemap=arelkbackgroundzh" target="_blank">GIS-Browser</a>
    </p>
</div>
</div>
<footer>© Statistisches Amt des Kantons Zürich</footer>
</body>
</html>
"""

    file_path = os.path.join(output_dir, f"{safe_name}.html")

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(html_content)

print("✅ Alle Steckbriefe im Zürich-Design erstellt.")

Erstelle Steckbrief für Aeugst am Albis...
Erstelle Steckbrief für Affoltern am Albis...
Erstelle Steckbrief für Bonstetten...
Erstelle Steckbrief für Hausen am Albis...
Erstelle Steckbrief für Hedingen...
Erstelle Steckbrief für Kappel am Albis...
Erstelle Steckbrief für Knonau...
Erstelle Steckbrief für Maschwanden...
Erstelle Steckbrief für Mettmenstetten...
Erstelle Steckbrief für Obfelden...
Erstelle Steckbrief für Ottenbach...
Erstelle Steckbrief für Rifferswil...
Erstelle Steckbrief für Stallikon...
Erstelle Steckbrief für Wettswil am Albis...
Erstelle Steckbrief für Benken (ZH)...
Erstelle Steckbrief für Berg am Irchel...
Erstelle Steckbrief für Buch am Irchel...
Erstelle Steckbrief für Dachsen...
Erstelle Steckbrief für Dorf...
Erstelle Steckbrief für Feuerthalen...
Erstelle Steckbrief für Flaach...
Erstelle Steckbrief für Flurlingen...
Erstelle Steckbrief für Henggart...
Erstelle Steckbrief für Kleinandelfingen...
Erstelle Steckbrief für Laufen-Uhwiesen...
Erstelle Steckbrief

### Steckbriefe für Bezirke

TBD: 
- Lösung für Bezirksfreies Gebiet (100)

In [ ]:
import json
import os
from urllib.request import urlopen
import folium
import requests
import xml.etree.ElementTree as ET

# -----------------------------
# 1. Bezirke laden
# -----------------------------
bezirke_url = "https://gebietsstammdaten.statistik.zh.ch/api/bezirke"

with urlopen(bezirke_url) as response:
    bezirke_data = json.load(response)

bezirke = bezirke_data["bezirke"]

output_dir = "../../docs/bezirke/"
os.makedirs(output_dir, exist_ok=True)

# -----------------------------
# 2. Für jeden Bezirk Steckbrief erzeugen
# -----------------------------
for g in bezirke:

    bezirk_code = g["bezirk_code"]
    bezirk_name = g["bezirk_name"]

    gemeinden_url = f"{bezirke_url}/{bezirk_code}"

    with urlopen(gemeinden_url) as response:
        gemeinden_data = json.load(response)

    gemeinden = gemeinden_data["gemeinden"]

    # -----------------------------
    # Gemeinden alphabetisch sortieren
    # -----------------------------
    gemeinden_sorted = sorted(gemeinden, key=lambda x: x["gemeinde_name"])

    # Anzahl Gemeinden
    anzahl_gemeinden = len(gemeinden_sorted)

    # -----------------------------
    # Gemeinden HTML erzeugen
    # -----------------------------
    gemeinden_str = '<div class="gemeinden-grid">'

    for gemeinde in gemeinden_sorted:
        code = gemeinde["gemeinde_code"]
        name = gemeinde["gemeinde_name"]

        gemeinden_str += f"""
        <a class="gemeinde-chip" href="../gemeinden/{code}.html" target="_blank" rel="noopener">
            {name}
        </a>
        """

    gemeinden_str += "</div>"

    print(f"Erstelle Steckbrief für Bezirk {bezirk_name}...")

    safe_name = bezirk_code
    if bezirk_code == 100:
        continue
    #Geodaten laden
    filter_encoded = f"%3CFilter%3E%3CPropertyIsEqualTo%3E%3CPropertyName%3Ebezirk%3C/PropertyName%3E%3CLiteral%3E{bezirk_name}%3C/Literal%3E%3C/PropertyIsEqualTo%3E%3C/Filter%3E"

    base_wfs = "https://maps.zh.ch/wfs/OGDZHWFS"

    geojson_link = f"{base_wfs}?SERVICE=WFS&VERSION=2.0.0&REQUEST=GetFeature&TYPENAMES=ms:ogd-0095_arv_basis_up_bezirke_f&outputFormat=geojson&FILTER={filter_encoded}"

    gml_link = f"{base_wfs}?SERVICE=WFS&VERSION=2.0.0&REQUEST=GetFeature&TYPENAMES=ms:ogd-0095_arv_basis_up_bezirke_f&outputFormat=gml3&FILTER={filter_encoded}"
    
    #Zentrierung anhand Koordinaten im GML für Link aufs Geoportal ZH erzeugen
    response = requests.get(gml_link)
    tree = ET.fromstring(response.content)

    ns = {
        "wfs": "http://www.opengis.net/wfs/2.0",
        "gml": "http://www.opengis.net/gml",
        "ms": "http://mapserver.gis.umn.edu/mapserver"
    }

    # gml parsen
    envelope = tree.find(".//wfs:boundedBy/gml:Envelope", ns)
    lower_corner = envelope.find("gml:lowerCorner", ns).text
    upper_corner = envelope.find("gml:upperCorner", ns).text

    lower_x, lower_y = map(float, lower_corner.split())
    upper_x, upper_y = map(float, upper_corner.split())

    x_center = ((lower_x + upper_x) / 2) - 3000 #Minus tausend 3000, aufgrund der Legende links. So wird es im Kartenbereich mittig.
    y_center = (lower_y + upper_y) / 2

    scale = 70000

    gis_link = f"https://geo.zh.ch/maps?x={x_center}&y={y_center}&scale={scale}&basemap=arelkbackgroundzh"
    # -----------------------------
    # HTML Steckbrief
    # -----------------------------
    html_content = f"""
<!DOCTYPE html>
<html lang="de">
<head>
<meta charset="UTF-8">
<title>Bezirk {bezirk_name}</title>

<style>

body {{
    font-family: Arial, sans-serif;
    margin: 0;
    background-color: #f4f6f9;
}}

.header {{
    background-color: #0076bd;
    color: white;
    padding: 20px 40px;
    display: flex;
    align-items: center;
}}

.header img {{
    height: 60px;
    margin-right: 20px;
    filter: brightness(0) invert(1);
}}

.header h1 {{
    margin: 0;
}}

.content {{
    padding: 40px;
}}

.card {{
    background: white;
    padding: 25px;
    border-radius: 12px;
    box-shadow: 0 4px 12px rgba(0,0,0,0.08);
    margin-bottom: 30px;
}}

.card h2 {{
    color: #0076bd;
    margin-top: 0;
}}

iframe {{
    width: 100%;
    height: 500px;
    border: none;
    border-radius: 12px;
}}

.download-btn {{
    background-color: #0076bd;
    color: white;
    padding: 12px 18px;
    border-radius: 6px;
    text-decoration: none;
    display: inline-block;
}}

.download-btn:hover {{
    background-color: #005fa3;
}}

.stat-box {{
    font-size: 18px;
    margin-bottom: 20px;
}}

.gemeinden-grid {{
    display: flex;
    flex-wrap: wrap;
    gap: 10px;
}}

.gemeinde-chip {{
    background-color: #eef3f8;
    color: #003D7A;
    padding: 8px 14px;
    border-radius: 20px;
    text-decoration: none;
    font-size: 14px;
    border: 1px solid #d0d7e2;
    transition: all 0.2s ease;
}}

.gemeinde-chip:hover {{
    background-color: #0076bd;
    color: white;
    border-color: #0076bd;
}}

footer {{
    text-align: center;
    padding: 20px;
    color: #666;
}}

</style>
</head>

<body>

<div class="header">
    <img src="../logo.svg" alt="Zürich Leu">
    <h1>Bezirk {bezirk_name}</h1>
</div>

<div class="content">


<div class="card">
<p><strong>Bezirk-Code:</strong> {bezirk_code}<br></p>
<p><strong>Anzahl Gemeinden:</strong> {anzahl_gemeinden}</p>
</div>

<div class="card">
<h2>Gemeinden des Bezirks</h2>

{gemeinden_str}

</div>

<div class="card">
<h2>Lage</h2>

<iframe src="https://map.geo.admin.ch/embed.html?ch.swisstopo.swissboundaries3d-bezirk-flaeche.fill={bezirk_code}"></iframe>

</div>

<div class="card">
    <h2>Geodaten</h2>
    <a class="download-btn" href="{gis_link}" target="_blank">Geoportal Kanton Zürich</a>
   <a class="download-btn" href="{geojson_link}" target="_blank">GeoJSON (WFS)</a>
<a class="download-btn" href="{gml_link}" target="_blank">GML</a>
</div>

<div class="card">
    <h2>Datenquellen / Links</h2>
    <p>    
        Gemeinden und Bezirke: 
        <a href="https://www.bfs.admin.ch/bfs/de/home/grundlagen/agvch.html" target="_blank">BFS – Amtliches Gemeindenverzeichnis</a><br>
        Raumplanungsregionen: 
        <a href="https://www.bfs.admin.ch/bfs/de/home/statistiken/querschnittsthemen/raeumliche-analysen/raeumliche-gliederungen/regionalpolitische-gliederungen.html" target="_blank">Raumplanungsregionen</a><br>
        Gebietsstammdaten ZH: 
        <a href="https://www.zh.ch/de/politik-staat/statistik-daten/datenkatalog.html#/datasets/3082@statistisches-amt-kanton-zuerich" target="_blank">Datenkatalog ZH</a><br>
Geoportal Kanton Zürich: 
<a href="https://geo.zh.ch/maps?x=2682088&y=1253620&scale=282230&basemap=arelkbackgroundzh" target="_blank">GIS-Browser</a>
    </p>
</div>
</div>
<footer>
© Statistisches Amt des Kantons Zürich
</footer>

</body>
</html>
"""

    file_path = os.path.join(output_dir, f"{safe_name}.html")

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(html_content)

print("✅ Alle Steckbriefe im Zürich-Design erstellt.")

Erstelle Steckbrief für Bezirk Bezirksfreies Gebiet ZH...
Erstelle Steckbrief für Bezirk Affoltern...
Erstelle Steckbrief für Bezirk Andelfingen...
Erstelle Steckbrief für Bezirk Bülach...
Erstelle Steckbrief für Bezirk Dielsdorf...
Erstelle Steckbrief für Bezirk Hinwil...
Erstelle Steckbrief für Bezirk Horgen...
Erstelle Steckbrief für Bezirk Meilen...
Erstelle Steckbrief für Bezirk Pfäffikon...
Erstelle Steckbrief für Bezirk Uster...
Erstelle Steckbrief für Bezirk Winterthur...
Erstelle Steckbrief für Bezirk Dietikon...
Erstelle Steckbrief für Bezirk Zürich...
✅ Alle Steckbriefe im Zürich-Design erstellt.


In [6]:
import json
import os
from urllib.request import urlopen
import folium
import requests
import xml.etree.ElementTree as ET

# -----------------------------
# 1. Bezirke laden
# -----------------------------
rpr_url = "https://gebietsstammdaten.statistik.zh.ch/api/raumplanungsregionen"

with urlopen(rpr_url) as response:
    rpr_data = json.load(response)

rpr = rpr_data["raumplanungsregionen"]

output_dir = "../../docs/raumplanungsregionen/"
os.makedirs(output_dir, exist_ok=True)

# -----------------------------
# 2. Für jeden Bezirk Steckbrief erzeugen
# -----------------------------
for g in rpr:

    rpr_code = g["raumplanungsregion_code"]
    rpr_name = g["raumplanungsregion_name"]

    gemeinden_url = f"{rpr_url}/{rpr_code}"

    with urlopen(gemeinden_url) as response:
        gemeinden_data = json.load(response)

    gemeinden = gemeinden_data["gemeinden"]

    # -----------------------------
    # Gemeinden alphabetisch sortieren
    # -----------------------------
    gemeinden_sorted = sorted(gemeinden, key=lambda x: x["gemeinde_name"])

    # Anzahl Gemeinden
    anzahl_gemeinden = len(gemeinden_sorted)

    # -----------------------------
    # Gemeinden HTML erzeugen
    # -----------------------------
    gemeinden_str = '<div class="gemeinden-grid">'

    for gemeinde in gemeinden_sorted:
        code = gemeinde["gemeinde_code"]
        name = gemeinde["gemeinde_name"]

        gemeinden_str += f"""
        <a class="gemeinde-chip" href="../gemeinden/{code}.html" target="_blank" rel="noopener">
            {name}
        </a>
        """

    gemeinden_str += "</div>"

    print(f"Erstelle Steckbrief für RPR {rpr_name}...")

    safe_name = rpr_code

    #Geodaten laden
    filter_encoded = f"%3CFilter%3E%3CPropertyIsEqualTo%3E%3CPropertyName%3Erpr_nr_bfs%3C/PropertyName%3E%3CLiteral%3E{rpr_code}%3C/Literal%3E%3C/PropertyIsEqualTo%3E%3C/Filter%3E"

    base_wfs = "https://maps.zh.ch/wfs/OGDZHWFS"

    geojson_link = f"{base_wfs}?SERVICE=WFS&VERSION=2.0.0&REQUEST=GetFeature&TYPENAMES=ms:ogd-0183_giszhpub_rp_raumplanungsregionen_f&outputFormat=geojson&FILTER={filter_encoded}"

    gml_link = f"{base_wfs}?SERVICE=WFS&VERSION=2.0.0&REQUEST=GetFeature&TYPENAMES=ms:ogd-0183_giszhpub_rp_raumplanungsregionen_f&outputFormat=gml3&FILTER={filter_encoded}"
    
    #Zentrierung anhand Koordinaten im GML für Link aufs Geoportal ZH erzeugen
    response = requests.get(gml_link)
    tree = ET.fromstring(response.content)

    ns = {
        "wfs": "http://www.opengis.net/wfs/2.0",
        "gml": "http://www.opengis.net/gml",
        "ms": "http://mapserver.gis.umn.edu/mapserver"
    }

    # gml parsen
    envelope = tree.find(".//wfs:boundedBy/gml:Envelope", ns)
    lower_corner = envelope.find("gml:lowerCorner", ns).text
    upper_corner = envelope.find("gml:upperCorner", ns).text

    lower_x, lower_y = map(float, lower_corner.split())
    upper_x, upper_y = map(float, upper_corner.split())

    x_center = ((lower_x + upper_x) / 2) - 3000 #Minus tausend 3000, aufgrund der Legende links. So wird es im Kartenbereich mittig.
    y_center = (lower_y + upper_y) / 2

    scale = 70000

    gis_link = f"https://geo.zh.ch/maps?x={x_center}&y={y_center}&scale={scale}&basemap=arelkbackgroundzh"
    # -----------------------------
    # HTML Steckbrief
    # -----------------------------
    html_content = f"""
<!DOCTYPE html>
<html lang="de">
<head>
<meta charset="UTF-8">
<title>Raumplanungsregion {rpr_name}</title>

<style>

body {{
    font-family: Arial, sans-serif;
    margin: 0;
    background-color: #f4f6f9;
}}

.header {{
    background-color: #0076bd;
    color: white;
    padding: 20px 40px;
    display: flex;
    align-items: center;
}}

.header img {{
    height: 60px;
    margin-right: 20px;
    filter: brightness(0) invert(1);
}}

.header h1 {{
    margin: 0;
}}

.content {{
    padding: 40px;
}}

.card {{
    background: white;
    padding: 25px;
    border-radius: 12px;
    box-shadow: 0 4px 12px rgba(0,0,0,0.08);
    margin-bottom: 30px;
}}

.card h2 {{
    color: #0076bd;
    margin-top: 0;
}}

iframe {{
    width: 100%;
    height: 500px;
    border: none;
    border-radius: 12px;
}}

.download-btn {{
    background-color: #0076bd;
    color: white;
    padding: 12px 18px;
    border-radius: 6px;
    text-decoration: none;
    display: inline-block;
}}

.download-btn:hover {{
    background-color: #005fa3;
}}

.stat-box {{
    font-size: 18px;
    margin-bottom: 20px;
}}

.gemeinden-grid {{
    display: flex;
    flex-wrap: wrap;
    gap: 10px;
}}

.gemeinde-chip {{
    background-color: #eef3f8;
    color: #003D7A;
    padding: 8px 14px;
    border-radius: 20px;
    text-decoration: none;
    font-size: 14px;
    border: 1px solid #d0d7e2;
    transition: all 0.2s ease;
}}

.gemeinde-chip:hover {{
    background-color: #0076bd;
    color: white;
    border-color: #0076bd;
}}

footer {{
    text-align: center;
    padding: 20px;
    color: #666;
}}

</style>
</head>

<body>

<div class="header">
    <img src="../logo.svg" alt="Zürich Leu">
    <h1>Raumplanungsregion {rpr_name}</h1>
</div>

<div class="content">


<div class="card">
<p><strong>Raumplanungsregion-Code:</strong> {rpr_code}<br></p>
<p><strong>Anzahl Gemeinden:</strong> {anzahl_gemeinden}</p>
</div>

<div class="card">
<h2>Gemeinden der Raumplanungsregion</h2>

{gemeinden_str}

</div>

<!---
<div class="card">
<h2>Lage</h2>

<iframe src="https://map.geo.admin.ch/embed.html?ch.swisstopo.swissboundaries3d-bezirk-flaeche.fill={rpr_code}"></iframe>

</div> --->

<div class="card">
    <h2>Geodaten</h2>
    <a class="download-btn" href="{gis_link}" target="_blank">Geoportal Kanton Zürich</a>
   <a class="download-btn" href="{geojson_link}" target="_blank">GeoJSON (WFS)</a>
<a class="download-btn" href="{gml_link}" target="_blank">GML</a>
</div>

<div class="card">
    <h2>Datenquellen / Links</h2>
    <p>    
        Gemeinden und Bezirke: 
        <a href="https://www.bfs.admin.ch/bfs/de/home/grundlagen/agvch.html" target="_blank">BFS – Amtliches Gemeindenverzeichnis</a><br>
        Raumplanungsregionen: 
        <a href="https://www.bfs.admin.ch/bfs/de/home/statistiken/querschnittsthemen/raeumliche-analysen/raeumliche-gliederungen/regionalpolitische-gliederungen.html" target="_blank">Raumplanungsregionen</a><br>
        Gebietsstammdaten ZH: 
        <a href="https://www.zh.ch/de/politik-staat/statistik-daten/datenkatalog.html#/datasets/3082@statistisches-amt-kanton-zuerich" target="_blank">Datenkatalog ZH</a><br>
Geoportal Kanton Zürich: 
<a href="https://geo.zh.ch/maps?x=2682088&y=1253620&scale=282230&basemap=arelkbackgroundzh" target="_blank">GIS-Browser</a>
    </p>
</div>
</div>
<footer>
© Statistisches Amt des Kantons Zürich
</footer>

</body>
</html>
"""

    file_path = os.path.join(output_dir, f"{safe_name}.html")

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(html_content)

print("✅ Alle Steckbriefe im Zürich-Design erstellt.")

Erstelle Steckbrief für RPR Stadt Zürich...
Erstelle Steckbrief für RPR Glattal...
Erstelle Steckbrief für RPR Furttal...
Erstelle Steckbrief für RPR Limmattal...
Erstelle Steckbrief für RPR Knonaueramt...
Erstelle Steckbrief für RPR Zimmerberg...
Erstelle Steckbrief für RPR Pfannenstiel...
Erstelle Steckbrief für RPR Zürcher Oberland...
Erstelle Steckbrief für RPR Winterthur und Umgebung...
Erstelle Steckbrief für RPR Weinland...
Erstelle Steckbrief für RPR Zürcher Unterland...
✅ Alle Steckbriefe im Zürich-Design erstellt.
